In [ ]:
# 0. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set workspace
DRIVE_ROOT = '/content/drive/MyDrive/evo1_vlabench_finetune'
!mkdir -p {DRIVE_ROOT}
print(f'📁 Workspace: {DRIVE_ROOT}')

In [ ]:
# 1. Install system dependencies
!apt-get update -qq
!apt-get install -y -qq git wget unzip ffmpeg \
  libosmesa6-dev patchelf libgl1-mesa-glx libegl1-mesa libgles2-mesa \
  build-essential libglfw3 libglfw3-dev
print('✅ System dependencies installed')

In [ ]:
# 2. Install Miniconda
import os
from pathlib import Path

CONDA_DIR = Path('/opt/conda')
CONDA_BIN = CONDA_DIR / 'bin' / 'conda'

if not CONDA_BIN.exists():
    print('📦 Installing Miniconda...')
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /opt/conda
    !rm /tmp/miniconda.sh
    print('✅ Miniconda installed')
else:
    print('✅ Miniconda already installed')

# Add conda to PATH
os.environ['PATH'] = f"{CONDA_DIR / 'bin'}:{os.environ['PATH']}"

# Initialize conda
!conda init bash --quiet
!conda config --set always_yes yes --quiet

!conda --version

In [ ]:
# 3. Headless rendering setup
import os
import warnings

os.environ['MUJOCO_GL'] = 'egl'
os.environ.pop('DISPLAY', None)
warnings.filterwarnings('ignore', message=r'.*X11.*')

print('MUJOCO_GL =', os.environ.get('MUJOCO_GL'))
print('DISPLAY =', os.environ.get('DISPLAY'))

In [ ]:
# 4. Create Evo-1 environment (torch 2.5.1, Python 3.10)
print('🔧 Creating Evo-1 environment...')

# Remove if exists
!conda env remove -n evo1 -y --quiet 2>/dev/null || true

# Create environment
!conda create -n evo1 python=3.10 -y --quiet

# Install PyTorch 2.5.1
!conda run -n evo1 pip install -q \
  torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# Install core dependencies
!conda run -n evo1 pip install -q \
  'numpy>=1.26.4,<2.0' pillow opencv-python-headless \
  transformers accelerate safetensors diffusers \
  einops timm scipy pyyaml tensorboard wandb \
  websockets huggingface_hub tqdm colorlog colorama

# Flash attention (optional)
!conda run -n evo1 pip install -q flash_attn 2>/dev/null || echo '⚠️ flash_attn skipped'

print('✅ Evo-1 environment ready')

# Verify
!conda run -n evo1 python -c "import torch; print(f'  torch: {torch.__version__}')"

In [ ]:
# 5. Create VLABench environment (lerobot 0.3.2, Python 3.10)
print('🔧 Creating VLABench environment...')

# Remove if exists
!conda env remove -n vlabench -y --quiet 2>/dev/null || true

# Create environment
!conda create -n vlabench python=3.10 -y --quiet

# Install LeRobot 0.3.2 with exact dependencies
!conda run -n vlabench pip install -q \
  'lerobot==0.3.2' \
  'draccus==0.10.0' \
  'datasets>=2.19.0,<=3.6.0' \
  'rerun-sdk>=0.21.0,<0.23.0' \
  'deepdiff>=7.0.1,<9.0.0' \
  pynput pyserial

# LeRobot dependencies
!conda run -n vlabench pip install -q \
  av zarr pillow jsonlines numba h5py pyarrow \
  imageio 'opencv-python-headless>=4.9.0'

# MuJoCo & simulation
!conda run -n vlabench pip install -q \
  'numpy>=1.24,<2.0' mujoco dm_control 'gymnasium==0.29.1'

# VLABench dependencies (including openai for GPT utils)
!conda run -n vlabench pip install -q \
  scipy tqdm colorlog colorama 'open3d>=0.18.0' openai

print('✅ VLABench environment ready')

# Verify
!conda run -n vlabench python -c "import lerobot; print(f'  lerobot: {lerobot.__version__}')"
!conda run -n vlabench python -c "from lerobot.common.datasets.lerobot_dataset import LeRobotDataset; print('  ✅ LeRobotDataset import works')"


In [ ]:
# 6. Clone repositories
from pathlib import Path

# Clone repos in /content/ (ephemeral but fast)
if not Path('/content/Evo-1/.git').exists():
    !git clone -q https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print('✅ Cloned Evo-1')
else:
    print('✅ Evo-1 already cloned')

if not Path('/content/VLABench/.git').exists():
    !git clone -q https://github.com/OpenMOSS/VLABench.git /content/VLABench
    print('✅ Cloned VLABench')
else:
    print('✅ VLABench already cloned')

if not Path('/content/rrt-algorithms/.git').exists():
    !git clone -q https://github.com/motion-planning/rrt-algorithms.git /content/rrt-algorithms
    print('✅ Cloned rrt-algorithms')
else:
    print('✅ rrt-algorithms already cloned')

In [ ]:
# 7. Install repos in their respective environments
from pathlib import Path

# Install rrt-algorithms in VLABench env
!conda run -n vlabench pip install -q -e /content/rrt-algorithms
print('✅ Installed rrt-algorithms in vlabench env')

# Install VLABench in VLABench env
!conda run -n vlabench pip install -q -e /content/VLABench
print('✅ Installed VLABench in vlabench env')

# Install Evo-1 requirements in Evo-1 env (filter out torch)
evo1_reqs = Path('/content/Evo-1/Evo_1/requirements.txt')
if evo1_reqs.exists():
    filtered_reqs = []
    for line in evo1_reqs.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#'):
            if not any(x in line.lower() for x in ['torch', 'torchvision', 'torchaudio']):
                filtered_reqs.append(line)
    
    temp_reqs = Path('/content/evo1_filtered_reqs.txt')
    temp_reqs.write_text('\n'.join(filtered_reqs))
    !conda run -n evo1 pip install -q -r {temp_reqs}
    print('✅ Installed Evo-1 requirements in evo1 env')

In [ ]:
# 8. Download VLABench assets
from pathlib import Path

assets_dir = Path('/content/VLABench/VLABench/assets')

if (assets_dir / 'obj').exists() and (assets_dir / 'scenes').exists():
    print('✅ VLABench assets already present')
else:
    print('📥 Downloading VLABench assets...')
    !conda run -n vlabench python /content/VLABench/scripts/download_assets.py
    print('✅ VLABench assets downloaded')

In [ ]:
# 9. Download Evo-1 checkpoint
from huggingface_hub import snapshot_download
from pathlib import Path

# Store in Drive for persistence
ckpt_dir = Path(f'{WORKSPACE}/checkpoints/libero')
ckpt_dir.mkdir(parents=True, exist_ok=True)
model_file = ckpt_dir / 'mp_rank_00_model_states.pt'

if model_file.exists() and model_file.stat().st_size > 1_000_000:
    print(f'✅ Checkpoint present ({model_file.stat().st_size/1e9:.2f} GB)')
else:
    print('📥 Downloading Evo-1 checkpoint...')
    snapshot_download(
        repo_id='MINT-SJTU/Evo1_LIBERO',
        local_dir=str(ckpt_dir),
        local_dir_use_symlinks=False,
    )
    print('✅ Checkpoint downloaded')

# Verify
required = ['config.json', 'norm_stats.json', 'mp_rank_00_model_states.pt']
missing = [f for f in required if not (ckpt_dir / f).exists()]
if missing:
    raise FileNotFoundError(f'Missing: {missing}')
print('✅ Checkpoint verified')

In [ ]:
# 10. Generate expert trajectories (VLABench environment) - PARALLEL
from pathlib import Path
import subprocess
import os
from datetime import datetime

HDF5_DIR = Path(f'{WORKSPACE}/hdf5_trajectories')
HDF5_DIR.mkdir(parents=True, exist_ok=True)

TASKS = ['select_toy', 'select_fruit', 'select_drink']
N_SAMPLES = 50  # Attempts per task
MAX_EPISODES = 20  # Stop after this many successes

print(f'🎯 Generating trajectories for {TASKS} (PARALLEL)')
print(f'   Samples: {N_SAMPLES}, Max episodes: {MAX_EPISODES}')

# Function to run trajectory generation for a single task
def generate_task(task):
    task_dir = HDF5_DIR / task
    task_dir.mkdir(exist_ok=True)
    
    # Check if already have data
    existing = list(task_dir.glob('*.hdf5'))
    if len(existing) >= MAX_EPISODES:
        print(f'  ✅ {task}: {len(existing)} trajectories already exist - skipping')
        return task, True, len(existing), ''
    
    print(f'  🔄 Starting {task}... (existing: {len(existing)})')
    
    # Use env command to inject MUJOCO_GL into conda environment
    cmd = [
        'conda', 'run', '--no-capture-output', '-n', 'vlabench',
        'env', 'MUJOCO_GL=egl',
        'python', '/content/VLABench/scripts/trajectory_generation.py',
        '--task-name', task,
        '--n-sample', str(N_SAMPLES),
        '--max-episode', str(MAX_EPISODES),
        '--save-dir', str(task_dir),
        '--robot', 'franka',
        '--record-video', 'True'
    ]
    
    start_time = datetime.now()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = (datetime.now() - start_time).total_seconds()
    
    # Count results
    results = list(task_dir.glob('*.hdf5'))
    success = result.returncode == 0
    
    status = '✅' if success else '❌'
    print(f'  {status} {task}: {len(results)} trajectories in {elapsed:.1f}s')
    
    return task, success, len(results), result.stderr if not success else ''

# Launch all tasks in parallel
print(f'\n🚀 Launching {len(TASKS)} parallel processes...\n')

from concurrent.futures import ThreadPoolExecutor, as_completed

results = {}
with ThreadPoolExecutor(max_workers=len(TASKS)) as executor:
    # Submit all tasks
    futures = {executor.submit(generate_task, task): task for task in TASKS}
    
    # Collect results as they complete
    for future in as_completed(futures):
        task, success, count, error = future.result()
        results[task] = {'success': success, 'count': count, 'error': error}

# Summary
print('\n' + '='*60)
print('TRAJECTORY GENERATION SUMMARY')
print('='*60)
total = 0
for task in TASKS:
    r = results[task]
    status = '✅' if r['success'] else '❌'
    print(f'{status} {task}: {r["count"]} trajectories')
    if r['error']:
        print(f'    Error: {r["error"][:200]}...')
    total += r['count']

print(f'\n✅ Total trajectories: {total}')

In [ ]:
# 10.5 DEBUG - Test trajectory generation with detailed output
from pathlib import Path
import subprocess
import os

print('🔍 DEBUGGING TRAJECTORY GENERATION\n')

# Test 1: Check environment variables in vlabench environment
print('=' * 60)
print('TEST 1: Environment variables in vlabench conda env')
print('=' * 60)
cmd = ['conda', 'run', '--no-capture-output', '-n', 'vlabench', 
       'python', '-c', 
       'import os; print("MUJOCO_GL:", os.environ.get("MUJOCO_GL")); print("DISPLAY:", os.environ.get("DISPLAY"))']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

# Test 2: Check if MuJoCo/GLFW can initialize with EGL
print('\n' + '=' * 60)
print('TEST 2: MuJoCo/GLFW initialization test')
print('=' * 60)
test_code = '''
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ.pop("DISPLAY", None)
print("Set MUJOCO_GL=egl")
print("Removed DISPLAY")

try:
    import mujoco
    print("✅ MuJoCo imported successfully")
    
    # Try to create a simple model
    xml = """
    <mujoco>
      <worldbody>
        <geom type="plane" size="1 1 0.1"/>
        <body pos="0 0 1">
          <geom type="sphere" size="0.1"/>
        </body>
      </worldbody>
    </mujoco>
    """
    model = mujoco.MjModel.from_xml_string(xml)
    print("✅ MuJoCo model created")
    
    data = mujoco.MjData(model)
    print("✅ MuJoCo data created")
    
    # Try to render
    renderer = mujoco.Renderer(model, height=480, width=640)
    print("✅ MuJoCo renderer created with EGL")
    renderer.close()
    print("✅ Renderer closed successfully")
    
except Exception as e:
    print(f"❌ Error: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
'''

cmd = ['conda', 'run', '--no-capture-output', '-n', 'vlabench',
       'env', 'MUJOCO_GL=egl',
       'python', '-c', test_code]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[:1000])

# Test 3: Run trajectory generation for ONE task with full output
print('\n' + '=' * 60)
print('TEST 3: Single trajectory generation (select_toy)')
print('=' * 60)

HDF5_DIR = Path(f'{WORKSPACE}/hdf5_trajectories')
task_dir = HDF5_DIR / 'select_toy'
task_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    'conda', 'run', '--no-capture-output', '-n', 'vlabench',
    'env', 'MUJOCO_GL=egl',
    'python', '/content/VLABench/scripts/trajectory_generation.py',
    '--task-name', 'select_toy',
    '--n-sample', '5',  # Just 5 attempts for debugging
    '--max-episode', '2',  # Just 2 successes needed
    '--save-dir', str(task_dir),
    '--robot', 'franka',
    '--record-video', 'True'
]

print('Command:', ' '.join(cmd))
print('\nRunning...\n')

result = subprocess.run(cmd, capture_output=True, text=True)

print('=' * 60)
print('RETURN CODE:', result.returncode)
print('=' * 60)

if result.stdout:
    print('\n--- STDOUT ---')
    print(result.stdout)

if result.stderr:
    print('\n--- STDERR ---')
    print(result.stderr)

# Check results
results = list(task_dir.glob('*.hdf5'))
print(f'\n--- RESULTS ---')
print(f'Files created: {len(results)}')
if results:
    for f in results:
        print(f'  - {f.name} ({f.stat().st_size} bytes)')


In [ ]:
# 11. Convert to LeRobot format (VLABench environment)
from pathlib import Path
import subprocess

LEROBOT_DIR = Path(f'{WORKSPACE}/lerobot_dataset')
DATASET_NAME = 'vlabench_finetune'

print(f'🔄 Converting to LeRobot format...')
print(f'   Input: {HDF5_DIR}')
print(f'   Output: {LEROBOT_DIR / DATASET_NAME}')

cmd = [
    'conda', 'run', '-n', 'vlabench', 'python',
    '/content/VLABench/scripts/convert_to_lerobot.py',
    '--dataset-name', DATASET_NAME,
    '--dataset-path', str(HDF5_DIR),
    '--task-list', *TASKS,
    '--max-files', '500',
]

result = subprocess.run(cmd, capture_output=True, text=True)
print('Return code:', result.returncode)
if result.stdout:
    print('\n--- Output ---')
    print(result.stdout)
if result.stderr:
    print('\n--- Errors ---')
    print(result.stderr)

if result.returncode == 0:
    print('\n✅ Conversion complete!')
else:
    print('\n❌ Conversion failed - check output above')

In [ ]:
# 12. Verify LeRobot dataset
!conda run -n vlabench python -c "
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset
from pathlib import Path

dataset_path = Path('{LEROBOT_DIR}') / '{DATASET_NAME}'
if dataset_path.exists():
    dataset = LeRobotDataset(str(dataset_path))
    print(f'✅ Dataset loaded')
    print(f'   Episodes: {dataset.num_episodes}')
    print(f'   Frames: {dataset.num_frames}')
    print(f'   Features: {list(dataset.features.keys())}')
else:
    print(f'❌ Dataset not found at {dataset_path}')
"

In [ ]:
# 13. Fine-tune Evo-1 (Evo-1 environment)
# TODO: Add Evo-1 fine-tuning script here
# This would use the LeRobot dataset to fine-tune the Evo-1 model

print('🔧 Fine-tuning step - to be implemented')
print('   Would load LeRobot dataset and fine-tune Evo-1 in evo1 environment')

In [ ]:
# 14. Environment info (debugging)
print('=== Evo-1 Environment ===')
!conda run -n evo1 python -c "
import torch
print(f'Python: {__import__(\"sys\").version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
"

print('\n=== VLABench Environment ===')
!conda run -n vlabench python -c "
import lerobot
import datasets
import draccus
print(f'Python: {__import__(\"sys\").version}')
print(f'LeRobot: {lerobot.__version__}')
print(f'Datasets: {datasets.__version__}')
print(f'Draccus: {draccus.__version__}')
"